In [ ]:
#import dataset to notebook

'''
#upload csv file to notebook
from google.colab import files
uploaded = files.upload()
'''
import pandas as pd


data = pd.read_csv("../data/raw/Customer-Data.csv")
data

In [ ]:
#display dataset columns
data.columns

In [ ]:
#display dataset info
data.info()

In [ ]:
#check for missing values
data.isnull().sum()

In [ ]:
#report and handle missing values

#defining columns groups
num_cols = ['Annual Income', 'Avg Monthly Spend']
cat_cols = ['Occupation', 'Education Level']

#count the num of missing values
num_missing = data[num_cols].isnull().sum()
cat_missing = data[cat_cols].isnull().sum()

if num_missing.sum() == 0 and cat_missing.sum() == 0:
  print("No missing values found in the dataset before cleaning")
else:
  #print num of missing values before cleaning
  print("Number of missing values in numerical columns before cleaning")
  print(num_missing)
  print("\nNumber of missing values in categorical columns before cleaning")
  print(cat_missing)


#numerical columns
data['Annual Income'] = data['Annual Income'].fillna(data['Annual Income'].median())
data['Avg Monthly Spend'] = data['Avg Monthly Spend'].fillna(data['Avg Monthly Spend'].median())

#categorical columns
data['Occupation'] = data['Occupation'].fillna(data['Occupation'].mode()[0])
data['Education Level'] = data['Education Level'].fillna(data['Education Level'].mode()[0])


#count num of missing values after cleaning
num_missing_after = data[num_cols].isnull().sum().sum()
cat_missing_after = data[cat_cols].isnull().sum().sum()

if num_missing_after == 0 and cat_missing_after == 0:
  print("No missing values found in the dataset after cleaning")
else:
  #print num of missing values after cleaning
  print("Number of missing values in numerical columns after cleaning")
  print(num_missing_after)
  print("\nNumber of missing values in categorical columns after cleaning")
  print(cat_missing_after)

In [ ]:
#remove duplicate records

duplicate_count = data.duplicated().sum()

#count num of dupe rows before cleaning
if duplicate_count == 0:
  print("No duplicate rows found in the dataset before cleaning")
else:
  print("Number of duplicate rows before cleaning: ")
  print(duplicate_count)

#check dupes based on CustomerID as that is a primary key
if 'CustomerID' in data.columns:
  id_dupe = data.duplicated(subset='CustomerID').sum()
  if id_dupe == 0:
    print("No duplicate rows found in the dataset based on CustomerID")
  else:
    print("Number of duplicate rows found in the dataset based on CustomerID: ")
    print(id_dupe)


#remove duplicates
data = data.drop_duplicates()
if 'CustomerID' in data.columns:
  data = data.drop_duplicates(subset='CustomerID')


#count num of dupe rows after cleaning
duplicate_count_after = data.duplicated().sum()
if duplicate_count_after == 0:
  print("No duplicate rows found in the dataset after cleaning")
else:
  print("Number of duplicate rows after cleaning: ")
  print(duplicate_count_after)



In [ ]:
#check and convert data types

print("Data types before conversion: ")
print(data.dtypes)

#convert numerical columns
data['Annual Income'] = pd.to_numeric(data['Annual Income'], errors='coerce')
data['Avg Monthly Spend'] = pd.to_numeric(data['Avg Monthly Spend'], errors='coerce')

print("\nData types after conversion:")
print(data.dtypes)



#check for conversion nulls
conversion_nulls = data[['Annual Income', 'Avg Monthly Spend']].isnull().sum()

if conversion_nulls.sum() == 0:
  print("\nNo conversion nulls found in the dataset")
else:
  print("\nNull conversion values found: ")
  print(conversion_nulls)

In [ ]:
#define numerical columns

num_columns = ['Annual Income', 'Avg Monthly Spend']
print("\nNumerical columns: ")
print(num_columns)

In [ ]:
#encode categorical vars

print("Dataset Shape ")
print(data.shape, "\n")

encoded_columns = [col for col in data.columns if 'Occupation_' in col or 'Education Level_' in col]

if len(encoded_columns) > 0:
  print("Categorical variables are already encoded")
  print("Number of Encoded columns: ", len(encoded_columns))
else:
  print("Categorical variables are not encoded")
  cat_columns = ['Occupation', 'Education Level']
  data = pd.get_dummies(data, columns=cat_columns, drop_first=True)
  print("Number of Encoded columns: ", len(data.columns))


In [ ]:
#dataset preview
display(data.head().T)

In [ ]:
#check for outliers

import matplotlib.pyplot as plt

plt.boxplot(data['Avg Monthly Spend'])
plt.title('Avg Monthly Spend Boxplot')
plt.ylabel('Avg Monthly Spend')
plt.show()

Q1 = data['Avg Monthly Spend'].quantile(0.25)
Q3 = data['Avg Monthly Spend'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data['Avg Monthly Spend'] < lower_bound) | (data['Avg Monthly Spend'] > upper_bound)]

print("\n")

if outliers.shape[0] == 0:
  print("There are no outliers in Avg Monthly Spend")
else:
  print("Number of outliers in Avg Monthly Spend:", outliers.shape[0])

In [ ]:
#remove outliers

before_removal = data.shape[0]

data = data[(data['Avg Monthly Spend'] >= lower_bound) & (data['Avg Monthly Spend'] <= upper_bound)]

after_removal = data.shape[0]

if before_removal == after_removal:
  print("No outliers were removed from the dataset")
else:
  print("Rows before outlier removal: ", before_removal)
  print("Rows after outlier removal: ", after_removal)
  print("Number of outliers removed from the dataset: ", before_removal - after_removal)


In [ ]:
###NOT USED


#feature scaling in order to ensure that all variables are comparable
#from sklearn.preprocessing import StandardScaler

#scaler = StandardScaler

#data[num_columns] = scaler().fit_transform(data[num_columns])

#print("Scaled Numerical Values Example\n")
#print(data[num_columns].head(4).T)

#print("\nScaled Numerical Values Mean: \n",data[num_columns].mean())
#print("\nScaled Numerical Values Standard Deviation: \n",data[num_columns].std())


In [ ]:
#export clean dataset

data.to_csv('../data/processed/Cleaned-Customer-Data.csv', index=False)

print("Cleaned dataset saved as Cleaned-Customer-Data.csv")
print("Final Dataset Shape:", data.shape)